In [6]:
import ast
import pandas as pd
df=pd.read_excel(r"C:\Users\91628\Downloads\raw_job_analysis (1).xlsx" )
df.head()

,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills
0,Senior Data Scientist,Senior Data Scientist Data and Analytics Perfo...,"Bennington, NE",via ZipRecruiter,Full-time,False,Sudan,2023-04-24 09:51:15,False,True,Sudan,year,128050.0,NaN,Cox Communications,"['sql', 'python', 'aws', 'pyspark', 'tableau',..."
1,Data Engineer,Data Engineer - MA,"Mesa, AZ",via Indeed,Full-time,False,Georgia,2023-03-13 12:51:23,True,True,United States,year,140000.0,NaN,Worldgate LLC,"['sql', 'nosql', 'java', 'python', 'kafka', 's..."
2,Senior Data Analyst,Supervisory Information Technology Specialist ...,"Alexandria, VA",via ZipRecruiter,Full-time,False,"New York, United States",2023-07-05 07:03:38,True,False,United States,year,156000.0,NaN,National Technical Information Service,NaN
3,Machine Learning Engineer,Machine Learning Research Scientist,"Pittsburgh, PA",via Ai-Jobs.net,Full-time,False,"Illinois, United States",2023-04-13 16:05:41,False,True,United States,year,140000.0,NaN,Bosch Group,"['pytorch', 'tensorflow']"
4,Data Scientist,"Data Scientist, AWS","Irving, TX",via Snagajob,Full-time and Part-time,False,"Texas, United States",2023-10-15 06:02:51,False,False,United States,hour,NaN,39.795002,"Presidio, Inc.","['python', 'r', 'sql', 'c', 'aws', 'gcp', 'big..."


In [7]:
df = df.drop_duplicates()

In [9]:
df = df.reset_index(drop=True)
df["job_id"] = df.index + 1
df.head()

,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_id
0,Senior Data Scientist,Senior Data Scientist Data and Analytics Perfo...,"Bennington, NE",via ZipRecruiter,Full-time,False,Sudan,2023-04-24 09:51:15,False,True,Sudan,year,128050.0,NaN,Cox Communications,"['sql', 'python', 'aws', 'pyspark', 'tableau',...",1
1,Data Engineer,Data Engineer - MA,"Mesa, AZ",via Indeed,Full-time,False,Georgia,2023-03-13 12:51:23,True,True,United States,year,140000.0,NaN,Worldgate LLC,"['sql', 'nosql', 'java', 'python', 'kafka', 's...",2
2,Senior Data Analyst,Supervisory Information Technology Specialist ...,"Alexandria, VA",via ZipRecruiter,Full-time,False,"New York, United States",2023-07-05 07:03:38,True,False,United States,year,156000.0,NaN,National Technical Information Service,NaN,3
3,Machine Learning Engineer,Machine Learning Research Scientist,"Pittsburgh, PA",via Ai-Jobs.net,Full-time,False,"Illinois, United States",2023-04-13 16:05:41,False,True,United States,year,140000.0,NaN,Bosch Group,"['pytorch', 'tensorflow']",4
4,Data Scientist,"Data Scientist, AWS","Irving, TX",via Snagajob,Full-time and Part-time,False,"Texas, United States",2023-10-15 06:02:51,False,False,United States,hour,NaN,39.795002,"Presidio, Inc.","['python', 'r', 'sql', 'c', 'aws', 'gcp', 'big...",5


In [10]:
df["job_platform"] = df["job_via"].str.replace("via ", "", regex=False)
df["job_platform"] = df["job_platform"].fillna("Not specified")

In [11]:
df["job_location"] = df["job_location"].fillna("Not specified")
df["job_schedule_type"] = df["job_schedule_type"].fillna("Not specified")

In [12]:
df["job_schedule_primary"] = (
    df["job_schedule_type"].str.split(",").str[0].str.split(" and ").str[0]
)

In [13]:
df["salary_year_est"] = df["salary_year_avg"]
is_hourly = df["salary_rate"] == "hour"
df.loc[is_hourly, "salary_year_est"] = df.loc[is_hourly, "salary_hour_avg"] * 2080

In [14]:
df["job_posted_date"] = pd.to_datetime(df["job_posted_date"])
df["job_posted_month"] = df["job_posted_date"].dt.to_period("M").astype(str)
df["job_posted_day"] = df["job_posted_date"].dt.day_name()

In [15]:
def parse_skills(text):
    if pd.isna(text):
        return []
    return ast.literal_eval(text)
 
df["job_skills_list"] = df["job_skills"].apply(parse_skills)
df["skill_count"] = df["job_skills_list"].apply(len)
 
print("Cleaned shape:", df.shape)
print(df.isna().sum())


Cleaned shape: (32671, 24)
job_title_short              0
job_title                    0
job_location                 0
job_via                     10
job_schedule_type            0
job_work_from_home           0
search_location              0
job_posted_date              0
job_no_degree_mention        0
job_health_insurance         0
job_country                  0
salary_rate                  0
salary_year_avg          10636
salary_hour_avg          22035
company_name                 0
job_skills                3187
job_id                       0
job_platform                 0
job_schedule_primary         0
salary_year_est              0
job_posted_month             0
job_posted_day               0
job_skills_list              0
skill_count                  0
dtype: int64


In [16]:

main_table = df.drop(columns=["job_skills_list"])

main_table.to_csv(r"E:\job\cleaned_file.csv", index=False)

In [21]:
skills_table = df[
    ["job_id", "job_title_short", "job_skills_list"]
].explode("job_skills_list")

skills_table = skills_table.rename(
    columns={"job_skills_list": "skill"}
)

skills_table = skills_table.dropna(subset=["skill"])

skills_table.to_csv(
    r"E:\job\job_skills_exploded.csv",
    index=False
)

In [17]:
main_table 


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,...,salary_hour_avg,company_name,job_skills,job_id,job_platform,job_schedule_primary,salary_year_est,job_posted_month,job_posted_day,skill_count
0,Senior Data Scientist,Senior Data Scientist Data and Analytics Perfo...,"Bennington, NE",via ZipRecruiter,Full-time,False,Sudan,2023-04-24 09:51:15,False,True,...,NaN,Cox Communications,"['sql', 'python', 'aws', 'pyspark', 'tableau',...",1,ZipRecruiter,Full-time,128050.000000,2023-04,Monday,7
1,Data Engineer,Data Engineer - MA,"Mesa, AZ",via Indeed,Full-time,False,Georgia,2023-03-13 12:51:23,True,True,...,NaN,Worldgate LLC,"['sql', 'nosql', 'java', 'python', 'kafka', 's...",2,Indeed,Full-time,140000.000000,2023-03,Monday,7
2,Senior Data Analyst,Supervisory Information Technology Specialist ...,"Alexandria, VA",via ZipRecruiter,Full-time,False,"New York, United States",2023-07-05 07:03:38,True,False,...,NaN,National Technical Information Service,NaN,3,ZipRecruiter,Full-time,156000.000000,2023-07,Wednesday,0
3,Machine Learning Engineer,Machine Learning Research Scientist,"Pittsburgh, PA",via Ai-Jobs.net,Full-time,False,"Illinois, United States",2023-04-13 16:05:41,False,True,...,NaN,Bosch Group,"['pytorch', 'tensorflow']",4,Ai-Jobs.net,Full-time,140000.000000,2023-04,Thursday,2
4,Data Scientist,"Data Scientist, AWS","Irving, TX",via Snagajob,Full-time and Part-time,False,"Texas, United States",2023-10-15 06:02:51,False,False,...,39.795002,"Presidio, Inc.","['python', 'r', 'sql', 'c', 'aws', 'gcp', 'big...",5,Snagajob,Full-time,82773.604126,2023-10,Sunday,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32666,Senior Data Scientist,Senior Data Scientist,"Palo Alto, CA",via Indeed,Full-time,False,"California, United States",2023-08-29 18:05:55,False,True,...,NaN,Navan,"['go', 'python', 'sql', 'numpy', 'pandas', 'te...",32667,Indeed,Full-time,191000.000000,2023-08,Tuesday,8
32667,Data Analyst,eCommerce Data Analyst | Hybrid Work | W2 Acce...,"Austin, TX",via LinkedIn,Contractor,False,"Texas, United States",2023-03-15 19:01:41,False,False,...,42.500000,"TalentBurst, an Inc 5000 company","['python', 'excel']",32668,LinkedIn,Contractor,88400.000000,2023-03,Wednesday,2
32668,Data Scientist,Clinical Data Visualization Specialist - Remote,Anywhere,via ZipRecruiter,Full-time,True,"California, United States",2023-12-18 16:02:34,False,True,...,55.000000,Avispa Technology,"['sas', 'sas', 'python']",32669,ZipRecruiter,Full-time,114400.000000,2023-12,Monday,3
32669,Data Analyst,Data Analyst/Report Writer 2,"Austin, TX",via Adzuna,Full-time,False,"Texas, United States",2023-05-04 07:01:58,True,False,...,55.000000,My3Tech,"['sas', 'sas', 'word', 'excel', 'sharepoint']",32670,Adzuna,Full-time,114400.000000,2023-05,Thursday,5


In [23]:
skills_table

,job_id,job_title_short,skill
0,1,Senior Data Scientist,sql
0,1,Senior Data Scientist,python
0,1,Senior Data Scientist,aws
0,1,Senior Data Scientist,pyspark
0,1,Senior Data Scientist,tableau
...,...,...,...
32669,32670,Data Analyst,sas
32669,32670,Data Analyst,sas
32669,32670,Data Analyst,word
32669,32670,Data Analyst,excel
